

### **📋 SESSION 1: From Raw Data to Clean Insights**

#### **Étape 1.1: Imports et Configuration**

Import the necessary libraries for data manipulation and visualization.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration for plots
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")

#### **Étape 1.2: Chargement des Données**

Load the CSV file (make sure `fake_news.csv` is in the same folder).

#### **Étape 2.1: Analyse de la Distribution de la Cible**

Check if the dataset is balanced between Real and Fake news.

In [ ]:
# Count plot for the target column 'Etiquette'
sns.countplot(data=df, x='Etiquette', palette='viridis')
plt.title('Distribution of Real vs Fake News')
plt.show()

# Print exact counts
print(df['Etiquette'].value_counts())

#### **Étape 2.2: Détection des Valeurs Manquantes**

Check for missing values (NaN).

In [ ]:
# Check for missing values
print(df.isnull().sum())

# Visualizing missing values
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.show()

# Strategy: Fill missing text with empty string (if any)
df = df.fillna('')

#### **Étape 2.3: Détection des Duplicatas**

Remove duplicate rows to prevent the model from memorizing them.

In [ ]:
print(f"Duplicates before: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Duplicates after: {df.duplicated().sum()}")

#### **Étape 3.1: Analyse des Partages**

See if Fake news gets more shares than Real news.

In [ ]:
# Boxplot to visualize distribution of shares
sns.boxplot(data=df, x='Etiquette', y='Nb_Partages', palette='Set2')
plt.yscale('log') # Log scale helps if some articles have huge shares
plt.title('Distribution of Shares by Type')
plt.show()

#### **Étape 3.2: Analyse de la Longueur du Texte**

Compare the length of articles.

In [ ]:
# Create temporary length feature
df['Body_Length'] = df['Corps_Texte'].str.len()

# Plot histogram
sns.histplot(data=df, x='Body_Length', hue='Etiquette', kde=True, palette='coolwarm')
plt.title('Article Length Distribution')
plt.show()

#### **Étape 3.3: Exploration des Sources**

Extract the domain name from the URL.

In [ ]:
# Extract domain (simple split)
# Assuming format is like "http://www.site.com/..."
df['Source_Domain'] = df['URL_Source'].apply(lambda x: x.split('/')[2] if isinstance(x, str) and len(x.split('/')) > 2 else 'Unknown')

# Show top sources for Fake news
print(df[df['Etiquette'] == 'Fake']['Source_Domain'].value_counts().head(10))

-----

### **📋 SESSION 2: The Art of Feature Engineering**

#### **Cell 18 (Likely Step 2.1: Title Features)**

The notebook implies starting with statistical features. Let's create features for the **Title**.

In [ ]:
# Word count and Character count for Title
df['Title_Word_Count'] = df['Titre'].apply(lambda x: len(str(x).split()))
df['Title_Char_Count'] = df['Titre'].apply(lambda x: len(str(x)))

#### **Étape 2.2: Features Statistiques du Corps de Texte**

Do the same for the **Body**.

In [ ]:
# Word count and Character count for Body
df['Body_Word_Count'] = df['Corps_Texte'].apply(lambda x: len(str(x).split()))
df['Body_Char_Count'] = df['Corps_Texte'].apply(lambda x: len(str(x)))

# Average word length (Characters / Words)
df['Body_Avg_Word_Length'] = df['Body_Char_Count'] / (df['Body_Word_Count'] + 1) # +1 to avoid division by zero

#### **Étape 2.3: Indicateurs de Clickbait**

Detect sensationnalist patterns in titles.

In [ ]:
def detect_clickbait(text):
    text = str(text).lower()
    score = 0
    if '!' in text: score += 1
    if '?' in text: score += 1
    if 'video' in text: score += 1
    if 'exclusif' in text: score += 1
    if text.isupper(): score += 2 # ALL CAPS TITLES
    return score

df['Clickbait_Score'] = df['Titre'].apply(detect_clickbait)

#### **Étape 2.4: Ratio Partages/Longueur (Bot Detection Proxy)**

Detect if short articles have suspiciously high shares.

In [ ]:
# Shares per character
df['Shares_Per_Char'] = df['Nb_Partages'] / (df['Body_Char_Count'] + 1)

#### **Étape 2.5: Log Transformation des Partages**

Normalize the `Nb_Partages` column as it likely has a "long tail".

In [ ]:
df['Log_Shares'] = np.log1p(df['Nb_Partages'])

#### **Étape 2.6: Bucketization (Catégorisation des Partages)**

Group shares into categories.

In [ ]:
labels = ['Low', 'Popular', 'Viral']
bins = [-1, 1000, 10000, np.inf] # Define your own thresholds based on data
df['Virality_Category'] = pd.cut(df['Nb_Partages'], bins=bins, labels=labels)

# Check distribution
print(df['Virality_Category'].value_counts())

#### **Étape 2.7: Encodage de la Cible**

Convert 'Real'/'Fake' to 0/1.

In [ ]:
# Map 'Fake' to 1 and 'Real' to 0 (or vice versa)
target_map = {'Fake': 1, 'Real': 0}
df['Target'] = df['Etiquette'].map(target_map)

print("Target encoding complete.")

#### **Étape 2.8: Sélection des Features**

Prepare the final X (features) and y (target) matrices. We drop text columns and keep numbers.

In [ ]:
# Select only numeric columns we created
features = ['Title_Word_Count', 'Title_Char_Count',
            'Body_Word_Count', 'Body_Char_Count', 'Body_Avg_Word_Length',
            'Clickbait_Score', 'Shares_Per_Char', 'Log_Shares']

X = df[features]
y = df['Target']

# Fill any remaining NaNs (e.g., from division by zero) with 0
X = X.fillna(0)

-----

### **📋 SESSION 3: Building & Trusting Your Model**

#### **Étape 3.1: Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

#### **Étape 3.2: Entraînement du Modèle**

Use a Random Forest Classifier.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained!")

#### **Étape 3.3 & Context (Evaluation)**

Calculate the F1 Score and Accuracy.

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.2f}")
print("\nFull Report:\n")
print(classification_report(y_test, y_pred))

#### **Étape 3.4: Matrice de Confusion**

Visualize the errors.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

#### **Étape 3.5: Feature Importance**

See which features mattered most.

In [ ]:
# Get feature importances
importances = model.feature_importances_
feature_names = X.columns

# Create a DataFrame
feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False)

# Plot
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance')
plt.show()

-----

### **🎁 Part 4: Bonus (Optional)**

**Bonus Task 1 (Keywords):**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Filtrer uniquement les Fake News
fake_titles = df[df['Etiquette'] == 'Fake']['Titre'].fillna('')

# 2. Initialiser le vectorizer (pour compter les mots)

vectorizer = CountVectorizer(stop_words='english', max_features=20)

# 3. Transformer et compter
X_counts = vectorizer.fit_transform(fake_titles)
sum_words = X_counts.sum(axis=0)

# 4. Associer mots et fréquences
words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

# 5. Afficher le Top 10
print("Top 10 des mots-clés dans les Fake News :")
for word, freq in words_freq[:10]:
    print(f"- {word}: {freq}")

**Bonus Task 2 (Keywords):**

In [ ]:
# 1. Définir les seuils (Hypothèses)
# "High Shares" = Top 20% des partages
share_threshold = df['Nb_Partages'].quantile(0.80)
# "Low Content" = Texte très court (ex: moins de 500 caractères)
length_threshold = 500

# 2. Identifier les suspects
bots = df[(df['Nb_Partages'] > share_threshold) & (df['Body_Char_Count'] < length_threshold)]

print(f"Nombre d'articles suspects détectés : {len(bots)}")

# 3. Visualisation
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Body_Char_Count', y='Nb_Partages', hue='Etiquette', alpha=0.6)

# Ajouter un rectangle ou des lignes pour montrer la zone suspecte
plt.axhline(y=share_threshold, color='r', linestyle='--', label='Seuil Partages Élevés')
plt.axvline(x=length_threshold, color='r', linestyle=':', label='Seuil Texte Court')
plt.title('Détection de Comportement "Bot-like" (Partages vs Longueur)')
plt.legend()
plt.show()

**Bonus Task 3 (Keywords):**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Préparer les données
# On utilise les mêmes features X que précédemment
# La cible (y) devient le nombre de partages
y_viral = df['Nb_Partages']

# Split train/test
X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X, y_viral, test_size=0.2, random_state=42)

# 2. Entraîner le modèle de Régression
regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train_v, y_train_v)

# 3. Évaluer
y_pred_v = regressor.predict(X_test_v)
mae = mean_absolute_error(y_test_v, y_pred_v)

print(f"Erreur Moyenne Absolue (MAE) : {mae:.0f} partages")
print(f"Cela signifie que le modèle se trompe en moyenne de {mae:.0f} partages par article.")

# Comparer Réel vs Prédit (pour les 5 premiers)
comparison = pd.DataFrame({'Réel': y_test_v, 'Prédit': y_pred_v})
print("\nAperçu des prédictions :")
print(comparison.head())

**Bonus Task 4 (Keywords)**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Vectoriser le texte (Transformer les mots en chiffres pour K-Means)
# On limite à 1000 mots pour la rapidité
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
X_text = tfidf.fit_transform(df['Corps_Texte'].fillna(''))

# 2. Appliquer K-Means pour trouver 4 sujets (clusters)
num_clusters = 4
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans.fit(X_text)

# 3. Ajouter le cluster trouvé au DataFrame original
df['Cluster_Sujet'] = kmeans.labels_

# 4. Afficher les mots-clés par cluster pour identifier le sujet
print("Mots-clés principaux par Cluster (Sujet) :")
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf.get_feature_names_out()

for i in range(num_clusters):
    print(f"\nCluster {i} : ", end='')
    for ind in order_centroids[i, :10]: # Top 10 mots
        print(f"{terms[ind]}, ", end='')
    print() 

# 5. Visualiser la répartition
sns.countplot(x='Cluster_Sujet', hue='Etiquette', data=df)
plt.title('Répartition des Fake News par Sujet (Cluster)')
plt.show()